# SVM v6 -- Optimized (Filtered + No Z-score + All Features + GridSearchCV)

Combines best of v2 and v5:
- Filtered .npz data (helps cross-subject normalization)
- No Z-score on windows (preserves amplitude info for feature extraction)
- Full 176 features: RMS + MAV + WL + ZC + SSC + Hist(10) + CWT(7)
- GridSearchCV to find optimal kernel and hyperparameters

## Imports

In [1]:
import sys
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.utils import resample

from config import (
    RANDOM_SEED, N_CLASSES, MODELS_DIR, FIGURES_DIR,
    SUBJECTS_INTERDAY_LONG, SUBJECTS_INTERDAY_SHORT,
    SUBJECTS_FATIGUE, SUBJECTS_NON_FATIGUE,
    GESTURE_CLASSES,
)
from src.experiment_runner import TEST_SUBJECTS, TRAIN_SUBJECTS, META
from src.data_splitter import load_windows_from_metadata
from src.feature_extraction import extract_features_batch
from src.evaluation import measure_latency, print_latency

np.random.seed(RANDOM_SEED)

## Helpers

In [2]:
def load_filtered(df, verbose=False):
    X, y = load_windows_from_metadata(df, verbose=verbose)
    return X, y


def featurize(X, scaler=None, fit=False):
    F = extract_features_batch(X)
    if fit:
        scaler = StandardScaler().fit(F)
    if scaler is not None:
        F = scaler.transform(F)
    return F, scaler

## Train with GridSearchCV

In [ ]:
train_meta = META[
    (META["subject"].isin(TRAIN_SUBJECTS)) & (META["session"] == 0)
]
fatigue_train = META[
    (META["subject"].isin(SUBJECTS_FATIGUE)) & (META["position"].isin([0, 1]))
]
train_combined = pd.concat([train_meta, fatigue_train]).drop_duplicates()

X_train, y_train = load_filtered(train_combined, verbose=True)
F_train, scaler = featurize(X_train, fit=True)
print(f"Windows: {X_train.shape}, Features: {F_train.shape}")

MAX_SAMPLES = 80000
if len(F_train) > MAX_SAMPLES:
    idx = resample(np.arange(len(y_train)), n_samples=MAX_SAMPLES,
                   random_state=RANDOM_SEED, stratify=y_train)
    F_sub, y_sub = F_train[idx], y_train[idx]
    print(f"Subsampled to {MAX_SAMPLES}")
else:
    F_sub, y_sub = F_train, y_train

param_grid = [
    {"kernel": ["linear"], "C": [0.1, 1.0, 10.0]},
    {"kernel": ["rbf"], "C": [0.1, 1.0, 10.0], "gamma": ["scale", "auto"]},
]

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
grid = GridSearchCV(
    SVC(random_state=RANDOM_SEED),
    param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1,
    verbose=1,
)
grid.fit(F_sub, y_sub)

svm = grid.best_estimator_
print(f"Best params: {grid.best_params_}")
print(f"Best CV accuracy: {grid.best_score_:.4f}")
print(f"Support vectors: {svm.n_support_.sum()}")

print("\nAll CV results:")
results = pd.DataFrame(grid.cv_results_)
for _, row in results.iterrows():
    print(f"  {row['params']} -> {row['mean_test_score']:.4f} (+/- {row['std_test_score']:.4f})")

Loading windows: 100%|██████████| 5790/5790 [00:02<00:00, 2863.85it/s]


Windows: (651972, 8, 50), Features: (651972, 176)
Subsampled to 80000
Fitting 3 folds for each of 9 candidates, totalling 27 fits


## Per-subject Sanity Check

In [ ]:
per_subject_acc = {}
all_subjects = sorted(META[(META["session"] == 0)]["subject"].unique())

for subj in all_subjects:
    subj_df = META[
        (META["subject"] == subj) & (META["session"] == 0) & (META["position"] == 0)
    ]
    tr_df = subj_df[subj_df["repetition"].isin([0, 1])]
    te_df = subj_df[subj_df["repetition"] == 2]
    if len(tr_df) == 0 or len(te_df) == 0:
        continue

    X_tr, y_tr = load_filtered(tr_df)
    X_te, y_te = load_filtered(te_df)
    F_tr, sc = featurize(X_tr, fit=True)
    F_te, _ = featurize(X_te, sc)

    m = SVC(**grid.best_params_, random_state=RANDOM_SEED)
    m.fit(F_tr, y_tr)
    per_subject_acc[subj] = accuracy_score(y_te, m.predict(F_te))

ps_accs = list(per_subject_acc.values())
print(f"Per-subject: mean={np.mean(ps_accs):.4f}, min={np.min(ps_accs):.4f}, max={np.max(ps_accs):.4f}")
print()
for subj, acc in sorted(per_subject_acc.items()):
    bar = "#" * int(acc * 40)
    print(f"  {subj:4s}: {acc:.3f} {bar}")

## S1-S5 Zero-shot

In [ ]:
def predict_zs(df):
    X, y = load_filtered(df)
    F, _ = featurize(X, scaler)
    return accuracy_score(y, svm.predict(F))

s1_test = META[
    (META["subject"].isin(TEST_SUBJECTS))
    & (META["session"] == 0) & (META["position"] == 0)
]
s1_zs = predict_zs(s1_test)

s2_test = META[
    (META["subject"].isin(TEST_SUBJECTS))
    & (META["session"] == 0) & (META["position"] > 0)
]
s2_zs = predict_zs(s2_test)

s3_test = META[
    (META["subject"].isin(TEST_SUBJECTS)) & (META["session"] == 0)
]
s3_zs = predict_zs(s3_test)

interday_train_subj = [
    s for s in SUBJECTS_INTERDAY_LONG + SUBJECTS_INTERDAY_SHORT
    if s in TRAIN_SUBJECTS
]
s4_test = META[
    (META["subject"].isin(interday_train_subj))
    & (META["session"] > 0) & (META["position"] == 0)
]
s4_zs = predict_zs(s4_test) if len(s4_test) > 0 else float("nan")

s5_test = META[
    (META["subject"].isin(SUBJECTS_FATIGUE)) & (META["position"] >= 2)
]
s5_zs = predict_zs(s5_test) if len(s5_test) > 0 else float("nan")

print("Zero-shot:")
print(f"  S1 Ideal:        {s1_zs:.4f}")
print(f"  S2 Shift:        {s2_zs:.4f}")
print(f"  S3 Inter-subj:   {s3_zs:.4f}")
print(f"  S4 Inter-day:    {s4_zs:.4f}")
print(f"  S5 Fatigue:      {s5_zs:.4f}")

## S1-S5 Calibrated

In [ ]:
def calibrate_per_subject(test_df, cal_reps=(0, 1), test_rep=2,
                       p0_only=False):
    accs = []
    for subj in sorted(test_df["subject"].unique()):
        sdf = test_df[test_df["subject"] == subj]
        cal_df = sdf[sdf["repetition"].isin(cal_reps)]
        te_df = sdf[sdf["repetition"] == test_rep]
        if p0_only:
            cal_df = cal_df[cal_df["position"] == 0]
        if len(cal_df) == 0 or len(te_df) == 0:
            continue
        X_c, y_c = load_filtered(cal_df)
        X_t, y_t = load_filtered(te_df)
        F_c, _ = featurize(X_c, scaler)
        F_t, _ = featurize(X_t, scaler)
        clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED, C=1.0)
        clf.fit(F_c, y_c)
        accs.append(accuracy_score(y_t, clf.predict(F_t)))
    return np.mean(accs) if accs else float("nan")


s1_cal = calibrate_per_subject(s1_test)
s2_cal = calibrate_per_subject(s2_test)
s3_cal = calibrate_per_subject(s3_test)

s4_accs = []
for subj in sorted(s4_test["subject"].unique()):
    sdf = s4_test[s4_test["subject"] == subj]
    for sess in sorted(sdf["session"].unique()):
        sess_df = sdf[sdf["session"] == sess]
        cal_df = sess_df[sess_df["repetition"].isin([0, 1])]
        te_df = sess_df[sess_df["repetition"] == 2]
        if len(cal_df) == 0 or len(te_df) == 0:
            continue
        X_c, y_c = load_filtered(cal_df)
        X_t, y_t = load_filtered(te_df)
        F_c, _ = featurize(X_c, scaler)
        F_t, _ = featurize(X_t, scaler)
        clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED, C=1.0)
        clf.fit(F_c, y_c)
        s4_accs.append(accuracy_score(y_t, clf.predict(F_t)))
s4_cal = np.mean(s4_accs) if s4_accs else float("nan")

s5_accs = []
for subj in sorted(s5_test["subject"].unique()):
    sdf = META[(META["subject"] == subj)]
    cal_df = sdf[sdf["position"].isin([0, 1])]
    te_df = sdf[sdf["position"] >= 2]
    if len(cal_df) == 0 or len(te_df) == 0:
        continue
    X_c, y_c = load_filtered(cal_df)
    X_t, y_t = load_filtered(te_df)
    F_c, _ = featurize(X_c, scaler)
    F_t, _ = featurize(X_t, scaler)
    clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED, C=1.0)
    clf.fit(F_c, y_c)
    s5_accs.append(accuracy_score(y_t, clf.predict(F_t)))
s5_cal = np.mean(s5_accs) if s5_accs else float("nan")

print("Calibrated:")
print(f"  S1 Ideal:        {s1_cal:.4f}")
print(f"  S2 Shift:        {s2_cal:.4f}")
print(f"  S3 Inter-subj:   {s3_cal:.4f}")
print(f"  S4 Inter-day:    {s4_cal:.4f}")
print(f"  S5 Fatigue:      {s5_cal:.4f}")

## S6: Combined Factor

In [ ]:
s6_test = META[
    (META["subject"].isin(TEST_SUBJECTS))
    & (META["session"] == 0) & (META["position"] > 0)
]
X_s6, y_s6 = load_filtered(s6_test)
F_s6, _ = featurize(X_s6, scaler)
s6_zs = accuracy_score(y_s6, svm.predict(F_s6))
print(f"S6 zero-shot: {s6_zs:.4f}")

s6_cal_accs = []
for subj in sorted(TEST_SUBJECTS):
    sdf = s6_test[s6_test["subject"] == subj]
    cal_df = sdf[sdf["repetition"].isin([0, 1])]
    te_df = sdf[sdf["repetition"] == 2]
    if len(cal_df) == 0 or len(te_df) == 0:
        continue
    X_c, y_c = load_filtered(cal_df)
    X_t, y_t = load_filtered(te_df)
    F_c, _ = featurize(X_c, scaler)
    F_t, _ = featurize(X_t, scaler)
    clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED, C=1.0)
    clf.fit(F_c, y_c)
    s6_cal_accs.append(accuracy_score(y_t, clf.predict(F_t)))

s6_cal = np.mean(s6_cal_accs) if s6_cal_accs else float("nan")
print(f"S6 calibrated avg: {s6_cal:.4f}")

## S7: Gradual Electrode Shift (h24-h29)

In [ ]:
shift_subjects = [f"h{i}" for i in range(24, 30)]
shift_data = META[(META["subject"].isin(shift_subjects)) & (META["session"] == 0)]

s7_acc = {}
for pos in sorted(shift_data["position"].unique()):
    pos_df = shift_data[shift_data["position"] == pos]
    X_p, y_p = load_filtered(pos_df)
    F_p, _ = featurize(X_p, scaler)
    s7_acc[pos] = accuracy_score(y_p, svm.predict(F_p))
    print(f"  Position {pos:2d}: {s7_acc[pos]:.4f}")

baseline = s7_acc.get(0, 0)
avg_shifted = np.mean([s7_acc[p] for p in s7_acc if p > 0])
print(f"\nP0 baseline: {baseline:.4f}, Avg shifted: {avg_shifted:.4f}, "
      f"Degradation: {(baseline - avg_shifted) * 100:.2f}%")

plt.figure(figsize=(8, 4))
plt.plot(list(s7_acc.keys()), list(s7_acc.values()), "o-", linewidth=2)
plt.axhline(y=baseline, color="gray", linestyle="--", alpha=0.5, label="P0 baseline")
plt.xlabel("Position")
plt.ylabel("Accuracy")
plt.title("SVM v6 -- Accuracy vs Electrode Shift (h24-h29)")
plt.ylim(0, 1)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(FIGURES_DIR / "svm_v6_shift_curve.png"), dpi=150)
plt.show()

## S8: Few-Shot Calibration Analysis

In [ ]:
subset_s8 = META[(META["session"] == 0) & (META["subject"].isin(TEST_SUBJECTS))]

configs = [
    ("Zero-shot",        [],     False, 0),
    ("1 rep, p0 only",   [0],    True,  7),
    ("1 rep, all pos",   [0],    False, 77),
    ("2 rep, p0 only",   [0, 1], True,  14),
    ("2 rep, all pos",   [0, 1], False, 154),
]

s8_results = {}
for label, reps, p0_only, n_trials in configs:
    cal_accs = []
    for subj in TEST_SUBJECTS:
        sdf = subset_s8[subset_s8["subject"] == subj]
        if len(reps) == 0:
            X_t, y_t = load_filtered(sdf)
            F_t, _ = featurize(X_t, scaler)
            cal_accs.append(accuracy_score(y_t, svm.predict(F_t)))
            continue
        if p0_only:
            cal_df = sdf[(sdf["repetition"].isin(reps)) & (sdf["position"] == 0)]
        else:
            cal_df = sdf[sdf["repetition"].isin(reps)]
        te_df = sdf[sdf["repetition"] == 2]
        if len(cal_df) == 0 or len(te_df) == 0:
            continue
        X_c, y_c = load_filtered(cal_df)
        X_t, y_t = load_filtered(te_df)
        F_c, _ = featurize(X_c, scaler)
        F_t, _ = featurize(X_t, scaler)
        clf = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED, C=1.0)
        clf.fit(F_c, y_c)
        cal_accs.append(accuracy_score(y_t, clf.predict(F_t)))
    s8_results[label] = np.mean(cal_accs) if cal_accs else float("nan")
    print(f"  {label:20s}: {s8_results[label]:.4f}  (~{n_trials} trials)")

## Latency

In [ ]:
sample_window = X_train[:1]

def svm_predict_single(x):
    f = extract_features_batch(x)
    f = scaler.transform(f)
    return svm.predict(f)

latency = measure_latency(svm_predict_single, sample_window, n_runs=500)
print_latency(latency, model_name="SVM v6 (feature extract + predict)")

## Summary

In [ ]:
print()
print("=" * 60)
print("  SVM v6 -- RESULTS")
print(f"  Best params: {grid.best_params_}")
print(f"  Best CV acc:  {grid.best_score_:.4f}")
print("=" * 60)
print(f"{'Scenario':<25s} {'Zero-shot':>10s} {'Calibrated':>12s} {'Delta':>10s}")
print("-" * 60)
pairs = [
    ("S1 Ideal", s1_zs, s1_cal),
    ("S2 Shift", s2_zs, s2_cal),
    ("S3 Inter-subject", s3_zs, s3_cal),
    ("S4 Inter-day", s4_zs, s4_cal),
    ("S5 Fatigue", s5_zs, s5_cal),
]
for name, zs, cal in pairs:
    d = cal - zs
    print(f"{name:<25s} {zs*100:>9.2f}% {cal*100:>11.2f}% {d*100:>+9.2f}%")
print(f"{'S6 Combined':<25s} {s6_zs*100:>9.2f}% {s6_cal*100:>11.2f}% {(s6_cal-s6_zs)*100:>+9.2f}%")
print(f"{'S7 Gradual (avg p1-10)':<25s} {avg_shifted*100:>9.2f}% {'--':>12s} {'--':>10s}")
for label, val in s8_results.items():
    if label == "Zero-shot":
        continue
    print(f"{'S8 ' + label:<25s} {'--':>10s} {val*100:>11.2f}% {'--':>10s}")
print("-" * 60)
print(f"Per-subject sanity check: mean={np.mean(ps_accs):.4f}")
print(f"Latency (p95):           {latency['p95_ms']:.2f} ms")
print("=" * 60)

## Save

In [ ]:
joblib.dump({"model": svm, "scaler": scaler, "best_params": grid.best_params_},
            MODELS_DIR / "svm_v6.pkl")
print("Saved svm_v6.pkl")